# Animals-10 Image Classification ML Pipeline

This notebook follows the same style as the lecture notebooks and builds an end-to-end **classification** pipeline using the **Animals-10** Kaggle dataset.

**Pipeline steps included**
1. Data collection, validation and preparation  
2. Exploratory Data Analysis (EDA)  
3. Building and training a CNN classification model  
4. Model evaluation and improvement  
5. Prediction on unseen test data and a single input image  

> Before running the notebook, download the Kaggle dataset and extract it so that the folder structure contains a `raw-img` directory with the class folders inside it.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
import tensorflow as tf
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import PIL
import itertools

sns.set_style('whitegrid')


## 1. Data collection

Set the dataset path to the extracted Kaggle dataset folder.

Expected structure example:

- `../datasets/animals10/raw-img/cane/...`
- `../datasets/animals10/raw-img/gatto/...`
- `../datasets/animals10/raw-img/elefante/...`


In [ ]:

dataset_path = '../datasets/animals10/raw-img'   # update if needed

class_folders = sorted(os.listdir(dataset_path))
print(class_folders)
print("Number of classes:", len(class_folders))


In [ ]:

filepaths = []
labels = []

for label in class_folders:
    class_path = os.path.join(dataset_path, label)

    if os.path.isdir(class_path):
        for filename in os.listdir(class_path):
            filepath = os.path.join(class_path, filename)
            filepaths.append(filepath)
            labels.append(label)

image_df = pd.DataFrame({
    'filepath': filepaths,
    'label': labels
})

image_df.head()


## 2. Data validation and preparation

In [ ]:

print("Dataset shape:", image_df.shape)
print("\nMissing values:")
print(image_df.isnull().sum())

print("\nDuplicate rows:", image_df.duplicated().sum())

image_df['exists'] = image_df['filepath'].apply(os.path.exists)
print("\nFile existence check:")
print(image_df['exists'].value_counts())

image_df = image_df[image_df['exists'] == True].drop(columns='exists')
print("\nCleaned dataset shape:", image_df.shape)


In [ ]:

label_counts = image_df['label'].value_counts().sort_index()
print(label_counts)


## 3. Exploratory Data Analysis (EDA)

In [ ]:

plt.figure(figsize=(12, 5))
sns.barplot(x=label_counts.index, y=label_counts.values)
plt.title("Number of images per class")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()


In [ ]:

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(16, 7))

for ax, label in zip(axes.flatten(), sorted(image_df['label'].unique())):
    sample_path = image_df[image_df['label'] == label]['filepath'].iloc[0]
    sample_image = PIL.Image.open(sample_path)
    ax.imshow(sample_image)
    ax.set_title(label)
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:

image_df['label'].value_counts(normalize=True).sort_index()


## 4. Train / validation / test split

In [ ]:

train_df, test_df = train_test_split(
    image_df,
    test_size=0.15,
    random_state=42,
    stratify=image_df['label']
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1765,   # gives roughly 70 / 15 / 15 overall
    random_state=42,
    stratify=train_df['label']
)

print("* Train set:", train_df.shape)
print("* Validation set:", val_df.shape)
print("* Test set:", test_df.shape)


In [ ]:

class_names = sorted(image_df['label'].unique())
label_to_index = {}

for index, label in enumerate(class_names):
    label_to_index[label] = index

print(class_names)
print(label_to_index)


## 5. Image loading function

We convert the images to grayscale to keep the workflow close to the lecture CNN notebooks and to make training lighter.


In [ ]:

def load_images(dataframe, image_size=(64, 64)):
    X = []
    y = []

    for _, row in dataframe.iterrows():
        image = PIL.Image.open(row['filepath']).convert('L')
        image = image.resize(image_size)
        np_image = np.array(image)

        X.append(np_image)
        y.append(label_to_index[row['label']])

    X = np.array(X)
    y = np.array(y)

    return X, y


In [ ]:

X_train, y_train = load_images(train_df, image_size=(64, 64))
X_val, y_val = load_images(val_df, image_size=(64, 64))
X_test, y_test = load_images(test_df, image_size=(64, 64))

print("* Train set:", X_train.shape, y_train.shape)
print("* Validation set:", X_val.shape, y_val.shape)
print("* Test set:", X_test.shape, y_test.shape)


In [ ]:

def check_images(dataset, dataset_name):
    '''
    Checks images for:
    * being an array
    * shape (64x64)
    * colour channel values
    * NaN values
    '''
    invalid_count = 0
    valid_count = 0

    for idx, image in enumerate(dataset):
        if not isinstance(image, np.ndarray):
            print(f"{dataset_name} - Index {idx}: Not a valid image array")
            invalid_count += 1
            continue

        if image.shape != (64, 64):
            print(f"{dataset_name} - Index {idx}: Incorrect shape {image.shape}")
            invalid_count += 1
            continue

        if image.min() < 0 or image.max() > 255:
            print(f"{dataset_name} - Index {idx}: Invalid pixel values")
            invalid_count += 1
            continue

        if np.isnan(image).any():
            print(f"{dataset_name} - Index {idx}: Contains NaN values")
            invalid_count += 1
            continue

        valid_count += 1

    print(f"\n{dataset_name} - Valid images: {valid_count}")
    print(f"{dataset_name} - Invalid images: {invalid_count}")


In [ ]:

check_images(X_train, "Train")
check_images(X_val, "Validation")
check_images(X_test, "Test")


## 6. Reshape, scale and encode labels

In [ ]:

X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2], 1)
X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], X_val.shape[2], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2], 1)

print(X_train.shape)


In [ ]:

X_train = X_train.astype("float32") / 255.0
X_val = X_val.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0


In [ ]:

n_labels = len(class_names)

y_train = to_categorical(y_train, num_classes=n_labels)
y_val = to_categorical(y_val, num_classes=n_labels)
y_test = to_categorical(y_test, num_classes=n_labels)

print(y_train.shape, y_val.shape, y_test.shape)


## 7. Baseline CNN model

In [ ]:

def build_tf_model(input_shape, n_labels):
    model = Sequential()

    model.add(Conv2D(filters=16, kernel_size=(3, 3), input_shape=input_shape, activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))

    model.add(Conv2D(filters=16, kernel_size=(3, 3), activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))

    model.add(Flatten())

    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.25))

    model.add(Dense(n_labels, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    return model


In [ ]:

early_stop = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=2)


In [ ]:

baseline_model = build_tf_model(input_shape=X_train.shape[1:], n_labels=n_labels)

baseline_model.fit(
    x=X_train,
    y=y_train,
    epochs=8,
    validation_data=(X_val, y_val),
    verbose=1,
    callbacks=[early_stop]
)


In [ ]:

history = pd.DataFrame(baseline_model.history.history)
history.head()


In [ ]:

sns.set_style("whitegrid")
history[['loss', 'val_loss']].plot(style='.-')
plt.title("Baseline Model Loss")
plt.show()

print("\n")

history[['accuracy', 'val_accuracy']].plot(style='.-')
plt.title("Baseline Model Accuracy")
plt.show()


In [ ]:

baseline_test_loss, baseline_test_accuracy = baseline_model.evaluate(X_test, y_test)
print("Baseline test loss:", baseline_test_loss)
print("Baseline test accuracy:", baseline_test_accuracy)


## 8. Confusion matrix and classification report

In [ ]:

def confusion_matrix_and_report(X, y, pipeline, label_map):
    '''
    Print confusion matrix and report, and plot heatmap
    '''

    prediction = pipeline.predict(X)
    prediction = np.argmax(prediction, axis=1)

    y = np.argmax(y, axis=1)

    cm = confusion_matrix(y_true=y, y_pred=prediction)

    print('---  Confusion Matrix  ---')
    print(pd.DataFrame(
        cm,
        columns=["Actual " + sub for sub in label_map],
        index=["Predicted " + sub for sub in label_map]
    ))
    print("\n")

    print('---  Classification Report  ---')
    print(classification_report(y, prediction, target_names=label_map), "\n")

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=label_map,
        yticklabels=label_map
    )

    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()


In [ ]:

def clf_performance(X_train, y_train,
                    X_test, y_test,
                    X_val, y_val,
                    pipeline,
                    label_map):
    '''
    Print classification performance
    '''
    print("#### Train Set ####\n")
    confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

    print("#### Validation Set ####\n")
    confusion_matrix_and_report(X_val, y_val, pipeline, label_map)

    print("#### Test Set ####\n")
    confusion_matrix_and_report(X_test, y_test, pipeline, label_map)


In [ ]:

clf_performance(
    X_train, y_train,
    X_test, y_test,
    X_val, y_val,
    baseline_model,
    label_map=class_names
)


## 9. Improved CNN model

Here we attempt a simple optimisation by increasing the number of filters and adding one more dense layer.


In [ ]:

def build_tf_model_2(input_shape, n_labels):
    model = Sequential()

    model.add(Conv2D(filters=32, kernel_size=(3, 3), input_shape=input_shape, activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))

    model.add(Conv2D(filters=32, kernel_size=(3, 3), activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))

    model.add(Flatten())

    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.25))

    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.25))

    model.add(Dense(n_labels, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    return model


In [ ]:

early_stop = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=2)


In [ ]:

optimized_model = build_tf_model_2(input_shape=X_train.shape[1:], n_labels=n_labels)

optimized_model.fit(
    x=X_train,
    y=y_train,
    epochs=10,
    validation_data=(X_val, y_val),
    verbose=1,
    callbacks=[early_stop]
)


In [ ]:

history_2 = pd.DataFrame(optimized_model.history.history)
history_2.head()


In [ ]:

sns.set_style("whitegrid")
history_2[['loss', 'val_loss']].plot(style='.-')
plt.title("Optimized Model Loss")
plt.show()

print("\n")

history_2[['accuracy', 'val_accuracy']].plot(style='.-')
plt.title("Optimized Model Accuracy")
plt.show()


In [ ]:

optimized_test_loss, optimized_test_accuracy = optimized_model.evaluate(X_test, y_test)

print("Baseline test accuracy:", baseline_test_accuracy)
print("Optimized test accuracy:", optimized_test_accuracy)


## 10. Final model evaluation

Use the optimized model if its test accuracy is better.


In [ ]:

model = optimized_model


In [ ]:

clf_performance(
    X_train, y_train,
    X_test, y_test,
    X_val, y_val,
    model,
    label_map=class_names
)


## 11. Prediction on unseen test image

In [ ]:

index = 0

single_image_path = test_df.iloc[index]['filepath']
actual_label = test_df.iloc[index]['label']

single_image = PIL.Image.open(single_image_path).convert('L')
single_image = single_image.resize((64, 64))

plt.figure(figsize=(5, 5))
plt.imshow(single_image, cmap='gray')
plt.title(f"Actual class: {actual_label}")
plt.axis('off')
plt.show()


In [ ]:

live_data = np.array(single_image)
live_data = live_data.reshape(1, 64, 64, 1)
live_data = live_data.astype("float32") / 255.0

print(live_data.shape)


In [ ]:

prediction_proba = model.predict(live_data)
prediction_proba


In [ ]:

prediction_class = np.argmax(prediction_proba, axis=1)
prediction_class


In [ ]:

predicted_label = class_names[prediction_class[0]]

print(f"Predicted label: {predicted_label}")
print(f"Actual label: {actual_label}")


In [ ]:

prob_per_class = pd.DataFrame(
    data=prediction_proba[0],
    columns=['Probability']
)

prob_per_class = prob_per_class.round(3)
prob_per_class['Results'] = class_names
prob_per_class


In [ ]:

fig = px.bar(
    prob_per_class,
    x='Results',
    y='Probability',
    range_y=[0, 1],
    width=800,
    height=400,
    template='seaborn'
)
fig.update_xaxes(type='category')
fig.show()


## 12. Optional: predict a new image

Replace the file path below with any image that was not used in training.


In [ ]:

new_image_path = single_image_path   # replace this with your own image path if required

new_image = PIL.Image.open(new_image_path).convert('L')
new_image = new_image.resize((64, 64))

plt.figure(figsize=(5, 5))
plt.imshow(new_image, cmap='gray')
plt.title("New image")
plt.axis('off')
plt.show()

new_live_data = np.array(new_image)
new_live_data = new_live_data.reshape(1, 64, 64, 1)
new_live_data = new_live_data.astype("float32") / 255.0

new_prediction_proba = model.predict(new_live_data)
new_prediction_class = np.argmax(new_prediction_proba, axis=1)

print("Predicted label:", class_names[new_prediction_class[0]])
